# A10 Appendix Cue Semantics and Intervention Analysis

LaTeX slot: `fig:appendix-cue-semantics`

This notebook is intentionally analysis-heavy rather than metric-heavy.

It asks three appendix questions:
1. How does the router redistribute attention across cue families when cue availability changes?
2. Does routing become less rich when we remove portable cue information?
3. Do semantic interventions hurt quality in a family-specific way rather than as a generic random perturbation?

In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path('/workspace/FeaturedMoE/writing/260418_final_exp_figure')
DATA_DIR = ROOT / 'data'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import slot_viz_helpers as viz
importlib.reload(viz)

from slot_viz_helpers import category_bar_line_plot, setup_style

setup_style()

cue_profile_df = pd.read_csv(DATA_DIR / '04_cue_family_profile.csv')
cue_stats_df = pd.read_csv(DATA_DIR / '04_cue_routing_stats.csv')
intervention_shift_df = pd.read_csv(DATA_DIR / '05_feature_intervention_shift.csv')
intervention_score_df = pd.read_csv(DATA_DIR / '05_feature_intervention_scores.csv')

family_order = ['memory', 'focus', 'tempo', 'exposure']
family_label_map = {
    'memory': 'Memory',
    'focus': 'Focus',
    'tempo': 'Tempo',
    'exposure': 'Exposure',
}
family_colors = {
    'memory': '#C17C74',
    'focus': '#5B7C99',
    'tempo': '#2A8C82',
    'exposure': '#D39C43',
}
setting_order = ['full', 'remove_category', 'remove_time', 'sequence_only_portable']
setting_label_map = {
    'full': 'Full cue bank',
    'remove_category': 'Without category cues',
    'remove_time': 'Without timing cues',
    'sequence_only_portable': 'Sequence-only portable',
}
intervention_order = ['full', 'shuffle_all', 'repeat_flatten', 'switch_boost', 'tempo_compress', 'popularity_spike']
intervention_label_map = {
    'full': 'Full model',
    'shuffle_all': 'Shuffle all cues',
    'repeat_flatten': 'Flatten repetition signal',
    'switch_boost': 'Boost switching signal',
    'tempo_compress': 'Compress tempo signal',
    'popularity_spike': 'Spike popularity signal',
}

In [ ]:
display(Markdown('### (a) Cue availability changes the family mix more than it changes the overall story'))
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.9), constrained_layout=True)
for axis, dataset in zip(axes, ['Beauty', 'Retail Rocket']):
    subset = cue_profile_df[cue_profile_df['dataset'] == dataset].copy()
    subset['expert_family'] = pd.Categorical(subset['expert_family'], categories=family_order, ordered=True)
    for setting in setting_order:
        setting_subset = subset[subset['cue_setting'] == setting].sort_values('expert_family')
        axis.plot(
            [family_label_map[value] for value in setting_subset['expert_family']],
            setting_subset['usage'],
            marker='o',
            linewidth=2.0,
            label=setting_label_map[setting],
        )
    axis.set_title(dataset)
    axis.set_xlabel('Dominant cue family')
    axis.set_ylabel('Average routing mass')
    axis.set_ylim(0.0, 0.55)
    axis.grid(axis='y', alpha=0.25)
axes[0].legend(frameon=False, loc='upper left')
fig.suptitle('(a) Portable-cue ablations shift family balance, but they do not erase the coarse routing semantics', y=1.04, fontsize=15)
plt.show()

In [ ]:
display(Markdown('### (b) Removing cue information usually narrows routing richness'))
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.9), constrained_layout=True)
for axis, dataset in zip(axes, ['KuaiRec', 'Retail Rocket']):
    subset = cue_stats_df[cue_stats_df['dataset'] == dataset].copy()
    category_bar_line_plot(
        subset,
        category_col='cue_setting',
        bar_col='effective_experts',
        line_col='route_entropy',
        ax=axis,
        title=dataset,
        ylabel='Effective experts',
        xlabel='Which cue information remains available to the router',
        line_label='Route entropy',
        order=setting_order,
        category_labels=setting_label_map,
        rotate=0,
    )
    axis.tick_params(axis='x', pad=8)
fig.suptitle('(b) Portable cues affect how broadly and how diffusely the router uses its experts', y=1.05, fontsize=15)
plt.show()

In [ ]:
display(Markdown('### (c) Semantic interventions degrade quality in family-specific ways'))
score_df = intervention_score_df.copy()
ndcg_df = score_df[score_df['metric'] == 'NDCG'].copy()
hr_df = score_df[score_df['metric'] == 'HR'].copy()
shift_df = intervention_shift_df.copy()

ndcg_df['intervention'] = pd.Categorical(ndcg_df['intervention'], categories=intervention_order, ordered=True)
hr_df['intervention'] = pd.Categorical(hr_df['intervention'], categories=intervention_order, ordered=True)
shift_df['intervention'] = pd.Categorical(shift_df['intervention'], categories=intervention_order[1:], ordered=True)
shift_df['expert_group'] = pd.Categorical(shift_df['expert_group'], categories=family_order, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.0), constrained_layout=True)
score_ax, heatmap_ax = axes

score_plot_df = ndcg_df[['intervention', 'value']].rename(columns={'value': 'ndcg20'}).copy()
score_plot_df['hr10'] = hr_df.sort_values('intervention')['value'].to_numpy()
category_bar_line_plot(
    score_plot_df,
    category_col='intervention',
    bar_col='ndcg20',
    line_col='hr10',
    ax=score_ax,
    title='Intervention quality impact',
    ylabel='NDCG@20',
    xlabel='How the semantic cue signal is perturbed',
    line_label='HR@10',
    order=intervention_order,
    category_labels=intervention_label_map,
    rotate=0,
)
score_ax.tick_params(axis='x', pad=8)

heatmap_data = shift_df.pivot(index='intervention', columns='expert_group', values='delta_mass').loc[intervention_order[1:], family_order]
image = heatmap_ax.imshow(heatmap_data.to_numpy(), cmap='coolwarm', aspect='auto', vmin=-0.22, vmax=0.22)
heatmap_ax.set_xticks(range(len(family_order)), [family_label_map[value] for value in family_order])
heatmap_ax.set_yticks(range(len(heatmap_data.index)), [intervention_label_map[value] for value in heatmap_data.index])
heatmap_ax.set_xlabel('Dominant family receiving routing mass')
heatmap_ax.set_ylabel('Semantic intervention')
heatmap_ax.set_title('Family-mass shift after intervention')
for row_idx, row_name in enumerate(heatmap_data.index):
    for col_idx, col_name in enumerate(heatmap_data.columns):
        value = heatmap_data.loc[row_name, col_name]
        heatmap_ax.text(col_idx, row_idx, f'{value:+.2f}', ha='center', va='center', fontsize=8.5, color='#1F2937')
colorbar = fig.colorbar(image, ax=heatmap_ax, fraction=0.046, pad=0.04)
colorbar.set_label('Routing-mass delta')
plt.show()